# 🚀 SMIRK Demo on Google Colab (Free GPU)

3D Facial Reconstruction with **free NVIDIA T4 GPU**!

---

## ⚙️ Step 1: Enable GPU

**Go to Runtime → Change runtime type → GPU (T4)**

In [ ]:
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"PyTorch: {torch.__version__}")
    print(f"CUDA: {torch.version.cuda}")
else:
    print("❌ Enable GPU: Runtime → Change runtime type → GPU")

## 📥 Step 2: Clone Repository

In [ ]:
!git clone https://github.com/chinnu2534/smirk_ccn.git
%cd smirk_ccn
print("✅ Repository cloned!")

## 📦 Step 3: Install PyTorch3D First

Install PyTorch3D before other dependencies:

In [ ]:
# Install PyTorch3D - try prebuilt wheel first, fallback to source
import sys
import torch

# Get version info
python_ver = f"py3{sys.version_info.minor}"
cuda_ver = torch.version.cuda.replace(".", "")
torch_ver = torch.__version__.split("+")[0].replace(".", "")

print(f"Python: {python_ver}, CUDA: cu{cuda_ver}, PyTorch: pyt{torch_ver}")

# Try multiple wheel versions
wheel_urls = [
    f"https://dl.fbaipublicfiles.com/pytorch3d/packaging/wheels/{python_ver}_cu{cuda_ver}_pyt{torch_ver}/download.html",
    "https://dl.fbaipublicfiles.com/pytorch3d/packaging/wheels/py310_cu121_pyt241/download.html",
    "https://dl.fbaipublicfiles.com/pytorch3d/packaging/wheels/py310_cu121_pyt231/download.html",
]

!pip install -q fvcore iopath

installed = False
for url in wheel_urls:
    print(f"Trying: {url}")
    result = !pip install -q --no-index --no-cache-dir pytorch3d -f {url} 2>&1
    try:
        import pytorch3d
        print(f"✅ PyTorch3D {pytorch3d.__version__} installed!")
        installed = True
        break
    except:
        continue

# Fallback: build from source
if not installed:
    print("Building PyTorch3D from source (takes ~5 min)...")
    !pip install -q "git+https://github.com/facebookresearch/pytorch3d.git"
    import pytorch3d
    print(f"✅ PyTorch3D {pytorch3d.__version__} installed from source!")

## 📦 Step 4: Install Other Dependencies

In [ ]:
# Install requirements with version flexibility
!pip install -q albumentations omegaconf scikit-learn scikit-image timm tqdm chumpy gdown pytorch_lightning
!pip install -q opencv-python opencv-contrib-python

# Install latest compatible mediapipe
!pip install -q mediapipe

print("✅ Dependencies installed!")

## 📥 Step 5: Download Models

In [ ]:
!bash quick_install.sh
print("✅ Models downloaded!")

## 🖼️ Step 6: Upload Test Image

In [ ]:
from google.colab import files
import shutil, os

os.makedirs('samples', exist_ok=True)
print("Upload a face image:")
uploaded = files.upload()

for f in uploaded.keys():
    shutil.move(f, f'samples/{f}')
    test_image = f'samples/{f}'
    print(f"✅ Saved: {test_image}")
    break

from IPython.display import Image, display
display(Image(filename=test_image, width=300))

## 🎯 Step 7: Run Demo

In [ ]:
!python demo.py --input_path {test_image} --out_path results/ --checkpoint pretrained_models/SMIRK_em1.pt --crop
print("✅ Done!")

## 📊 Step 8: View Results

In [ ]:
from IPython.display import Image, display
import os

if os.path.exists('results'):
    for f in sorted(os.listdir('results')):
        if f.endswith(('.png', '.jpg')):
            print(f"📸 {f}")
            display(Image(filename=f'results/{f}', width=800))
else:
    print("No results found")

## 💾 Download Results

In [ ]:
import shutil
from google.colab import files
if os.path.exists('results'):
    shutil.make_archive('results', 'zip', 'results')
    files.download('results.zip')